In [ ]:
import plotly.graph_objects as go
import trimesh
from rasterio.enums import Resampling
from rasterio.windows import from_bounds
from pyproj import Transformer
import numpy as np
from campaign import *
from cir import *
from constants import *
from diffraction import *
from em_core import *
from engine_hybrid import *
from engine_image import *
from engine_image_parallel import *
from engine_sbr import *
from geometry import *
from legacy import *
from terrain_io import *
from timing import *
from viz import *
from scenarios import *
import plotly.io as pio
from pyproj import CRS
import spiceypy as spice
import pickle
from polyscope_live import launch_live
pio.renderers.default = 'browser'

spice.furnsh("/Users/pierazhi/Desktop/Multipath/Multipath/Refactored/kernels/naif0012.tls")
spice.furnsh("/Users/pierazhi/Desktop/Multipath/Multipath/Refactored/kernels/moon_de440_220930.tf")
spice.furnsh("/Users/pierazhi/Desktop/Multipath/Multipath/Refactored/kernels/moon_pa_de440_200625.bpc")
spice.furnsh("/Users/pierazhi/Desktop/Multipath/Multipath/Refactored/kernels/pck00011.tpc")
R_LOLA_M = 1_737_400.0


def _as_pyproj_crs(crs):
    """
    Convert rasterio CRS or pyproj CRS to pyproj CRS.
    """
    return CRS.from_user_input(crs)


def _get_dem_crs(src):
    if src.crs is None:
        raise ValueError("The DEM has no CRS.")
    return _as_pyproj_crs(src.crs)


def _get_lunar_geographic_crs(dem_crs):
    """
    Extract the geographic lunar CRS from the projected DEM CRS.
    """
    dem_crs = _as_pyproj_crs(dem_crs)

    geo_crs = dem_crs.geodetic_crs

    if geo_crs is None:
        raise ValueError("Could not extract geographic CRS from DEM CRS.")

    return geo_crs


def _get_moon_radius_from_crs(dem_crs):
    """
    Extract lunar reference radius from CRS.

    For your CRS, this should return 1737400 m.
    """
    dem_crs = _as_pyproj_crs(dem_crs)
    return float(dem_crs.ellipsoid.semi_major_metre)


def _pixel_center_xy(transform, rows, cols):
    """
    Convert raster row/col indices to projected x/y coordinates at pixel centers.
    Works also if the affine transform has rotation terms.
    """
    x = (
        transform.c
        + (cols + 0.5) * transform.a
        + (rows + 0.5) * transform.b
    )

    y = (
        transform.f
        + (cols + 0.5) * transform.d
        + (rows + 0.5) * transform.e
    )

    return x, y

def inspect_dem(dem_path):
    with rasterio.open(dem_path) as src:
        print("CRS:")
        print(src.crs)
        print()

        print("Size:")
        print(f"width  = {src.width}")
        print(f"height = {src.height}")
        print()

        print("Resolution [m/px]:")
        print(src.res)
        print()

        print("Scene Size:")
        print(f"{src.width * src.res[0] * 1e-3} x {src.height * src.res[1] * 1e-3} km")
        print()

        print("Bounds [projected meters]:")
        print(src.bounds)
        print()

        print("Transform:")
        print(src.transform)
        print()

        print("NoData:")
        print(src.nodata)

def visualize_dem_2d(
    dem_path,
    downsample_factor=20,
    colorscale="Earth_r",
    clip_percentiles=(2, 98),
    title="LOLA DEM 2D preview",
):
    with rasterio.open(dem_path) as src:
        out_h = max(2, src.height // downsample_factor)
        out_w = max(2, src.width // downsample_factor)

        dem = src.read(
            1,
            out_shape=(out_h, out_w),
            resampling=Resampling.nearest,
        ).astype(float)

        transform = src.transform * src.transform.scale(
            src.width / out_w,
            src.height / out_h,
        )

        nodata = src.nodata

    if nodata is not None:
        dem[dem == nodata] = np.nan

    rows, cols = np.indices(dem.shape)
    x_m, y_m = _pixel_center_xy(transform, rows, cols)

    zmin = zmax = None
    if clip_percentiles is not None:
        zmin = float(np.nanpercentile(dem, clip_percentiles[0]))
        zmax = float(np.nanpercentile(dem, clip_percentiles[1]))

    fig = go.Figure(data=go.Heatmap(
        x=x_m[0, :] / 1000.0,
        y=y_m[:, 0] / 1000.0,
        z=dem,
        zmin=zmin,
        zmax=zmax,
        colorscale=colorscale,
        colorbar=dict(title="Elevation [m]"),
        hovertemplate=(
            "X: %{x:.3f} km<br>"
            "Y: %{y:.3f} km<br>"
            "Elevation: %{z:.1f} m"
            "<extra></extra>"
        ),
    ))

    fig.update_layout(
        title=title,
        xaxis=dict(
            title="Polar stereographic X [km]",
            scaleanchor="y",
            scaleratio=1,
        ),
        yaxis=dict(title="Polar stereographic Y [km]"),
        width=900,
        height=800,
    )

    fig.show()

def latlon_to_dem_xy(lat_deg, lon_deg, dem_crs):
    """
    Lunar lat/lon -> DEM projected x/y meters.
    """
    geo_crs = _get_lunar_geographic_crs(dem_crs)

    transformer = Transformer.from_crs(
        geo_crs,
        dem_crs,
        always_xy=True,
    )

    x_m, y_m = transformer.transform(lon_deg, lat_deg)
    return x_m, y_m

def selection_to_bounds(
    dem_path,
    method,
    *,
    center_lat=None,
    center_lon=None,
    center_x_km=None,
    center_y_km=None,
    side_km=None,
    vertices_latlon=None,
    vertices_xy_km=None,
    xy_km_is_offset_from_raster_center=False,
):
    """
    Return rectangular crop bounds in DEM projected meters:
        left, bottom, right, top

    Supported methods:
        "center_latlon"
        "center_xy_km"
        "vertices_latlon"
        "vertices_xy_km"

    For xy_km methods:
        default = absolute polar stereographic km
        optional = offset from raster center, matching your older function style
    """

    with rasterio.open(dem_path) as src:
        dem_crs = _get_dem_crs(src)
        bounds = src.bounds

        raster_center_x = 0.5 * (bounds.left + bounds.right)
        raster_center_y = 0.5 * (bounds.bottom + bounds.top)

    if method == "center_latlon":
        if center_lat is None or center_lon is None or side_km is None:
            raise ValueError("center_latlon requires center_lat, center_lon, side_km.")

        cx, cy = latlon_to_dem_xy(center_lat, center_lon, dem_crs)

        half = 0.5 * side_km * 1000.0
        left = cx - half
        right = cx + half
        bottom = cy - half
        top = cy + half

    elif method == "center_xy_km":
        if center_x_km is None or center_y_km is None or side_km is None:
            raise ValueError("center_xy_km requires center_x_km, center_y_km, side_km.")

        cx = center_x_km * 1000.0
        cy = center_y_km * 1000.0

        if xy_km_is_offset_from_raster_center:
            cx = raster_center_x + cx
            cy = raster_center_y + cy

        half = 0.5 * side_km * 1000.0
        left = cx - half
        right = cx + half
        bottom = cy - half
        top = cy + half

    elif method == "vertices_latlon":
        if vertices_latlon is None or len(vertices_latlon) != 4:
            raise ValueError("vertices_latlon requires exactly 4 points: [(lat, lon), ...].")

        xs = []
        ys = []

        for lat, lon in vertices_latlon:
            x, y = latlon_to_dem_xy(lat, lon, dem_crs)
            xs.append(x)
            ys.append(y)

        left = min(xs)
        right = max(xs)
        bottom = min(ys)
        top = max(ys)

    elif method == "vertices_xy_km":
        if vertices_xy_km is None or len(vertices_xy_km) != 4:
            raise ValueError("vertices_xy_km requires exactly 4 points: [(x_km, y_km), ...].")

        xs = np.array([p[0] for p in vertices_xy_km], dtype=float) * 1000.0
        ys = np.array([p[1] for p in vertices_xy_km], dtype=float) * 1000.0

        if xy_km_is_offset_from_raster_center:
            xs = raster_center_x + xs
            ys = raster_center_y + ys

        left = float(np.min(xs))
        right = float(np.max(xs))
        bottom = float(np.min(ys))
        top = float(np.max(ys))

    else:
        raise ValueError(
            "method must be one of: center_latlon, center_xy_km, vertices_latlon, vertices_xy_km"
        )

    return left, bottom, right, top

def crop_dem_by_bounds(
    dem_path,
    output_path,
    bounds_m,
):
    """
    Crop a GeoTIFF using projected bounds in the DEM CRS.

    bounds_m = (left, bottom, right, top)
    """

    left, bottom, right, top = bounds_m

    with rasterio.open(dem_path) as src:
        raw_window = from_bounds(
            left,
            bottom,
            right,
            top,
            transform=src.transform,
        )

        window = raw_window.round_offsets().round_lengths()

        full_window = Window(0, 0, src.width, src.height)
        window = window.intersection(full_window)

        if window.width <= 0 or window.height <= 0:
            raise ValueError("Crop window is outside the DEM bounds.")

        data = src.read(1, window=window)

        new_transform = rasterio.windows.transform(window, src.transform)

        meta = src.meta.copy()
        meta.update({
            "height": int(window.height),
            "width": int(window.width),
            "transform": new_transform,
        })

    with rasterio.open(output_path, "w", **meta) as dst:
        dst.write(data, 1)

    print(f"Saved cropped DEM: {output_path}")
    print(
        f"Window: row={window.row_off}, col={window.col_off}, "
        f"height={window.height}, width={window.width}"
    )

    return output_path, window

def crop_dem_by_bounds(
    dem_path,
    output_path,
    bounds_m,
):
    """
        Crop a GeoTIFF using projected bounds in the DEM CRS.

        bounds_m = (left, bottom, right, top)

        # Centered square by lat/lon
    bounds_m = selection_to_bounds(
        dem_path,
        method="center_latlon",
        center_lat=-89.67,
        center_lon=129.78,
        side_km=25.0,
    )

    # Centered square by projected km
    bounds_m = selection_to_bounds(
        dem_path,
        method="center_xy_km",
        center_x_km=0.0,
        center_y_km=0.0,
        side_km=25.0,
    )

    # Four lat/lon vertices, rectangular envelope
    bounds_m = selection_to_bounds(
        dem_path,
        method="vertices_latlon",
        vertices_latlon=[
            (-89.8, 120.0),
            (-89.8, 140.0),
            (-89.5, 140.0),
            (-89.5, 120.0),
        ],
    )

    # Four projected-km vertices, rectangular envelope
    bounds_m = selection_to_bounds(
        dem_path,
        method="vertices_xy_km",
        vertices_xy_km=[
            (-10.0, -10.0),
            ( 10.0, -10.0),
            ( 10.0,  10.0),
            (-10.0,  10.0),
        ],
    )
    """

    left, bottom, right, top = bounds_m

    with rasterio.open(dem_path) as src:
        raw_window = from_bounds(
            left,
            bottom,
            right,
            top,
            transform=src.transform,
        )

        window = raw_window.round_offsets().round_lengths()

        full_window = Window(0, 0, src.width, src.height)
        window = window.intersection(full_window)

        if window.width <= 0 or window.height <= 0:
            raise ValueError("Crop window is outside the DEM bounds.")

        data = src.read(1, window=window)

        new_transform = rasterio.windows.transform(window, src.transform)

        meta = src.meta.copy()
        meta.update({
            "height": int(window.height),
            "width": int(window.width),
            "transform": new_transform,
        })

    with rasterio.open(output_path, "w", **meta) as dst:
        dst.write(data, 1)

    print(f"Saved cropped DEM: {output_path}")
    print(
        f"Window: row={window.row_off}, col={window.col_off}, "
        f"height={window.height}, width={window.width}"
    )

    return output_path, window

def dem_to_pa_mesh(
    dem_path,
    *,
    target_resolution_m=None,
    resampling_method="nearest",
    utc_time="2026-06-22T00:00:00",
):
    """
    Convert a lunar South-Pole polar stereographic DEM to a triangular mesh
    in the MOON_PA frame.

    Parameters
    ----------
    dem_path : str
        Path to the cropped or full DEM GeoTIFF.

    target_resolution_m : float or None
        Desired mesh grid spacing in meters.

        None:
            use native DEM resolution.

        Equal to native resolution:
            unchanged sampling.

        Larger than native resolution:
            coarser mesh, e.g. 20 m -> 100 m.

        Smaller than native resolution:
            interpolated finer mesh, e.g. 20 m -> 10 m.

    resampling_method : str
        One of:
            "nearest", "bilinear", "cubic", "cubic_spline",
            "lanczos", "average"

        For physically faithful native DEM:
            use "nearest".

        For smoother visualization:
            use "bilinear" or "cubic".

        For coarsening:
            use "average" or "bilinear".

    utc_time : str
        Epoch used for SPICE ME -> PA rotation.

    Returns
    -------
    mesh_pa : trimesh.Trimesh
        Triangular terrain mesh in MOON_PA coordinates.

    aux : dict
        Useful arrays and metadata.
    """

    resampling_methods = {
        "nearest": Resampling.nearest,
        "bilinear": Resampling.bilinear,
        "cubic": Resampling.cubic,
        "cubic_spline": Resampling.cubic_spline,
        "lanczos": Resampling.lanczos,
        "average": Resampling.average,
    }

    if resampling_method not in resampling_methods:
        raise ValueError(
            f"Unknown resampling_method='{resampling_method}'. "
            f"Choose one of {list(resampling_methods.keys())}."
        )

    resampling = resampling_methods[resampling_method]

    # ------------------------------------------------------------
    # 1. Read DEM, optionally resampled to target_resolution_m
    # ------------------------------------------------------------
    with rasterio.open(dem_path) as src:
        if src.crs is None:
            raise ValueError("DEM has no CRS. Cannot convert stereographic coordinates to lat/lon.")

        dem_crs = CRS.from_user_input(src.crs)

        geo_crs = dem_crs.geodetic_crs
        if geo_crs is None:
            raise ValueError("Could not extract lunar geographic CRS from DEM CRS.")

        r_moon = float(dem_crs.ellipsoid.semi_major_metre)

        native_w = src.width
        native_h = src.height

        native_res_x = abs(src.res[0])
        native_res_y = abs(src.res[1])

        if target_resolution_m is None:
            out_w = native_w
            out_h = native_h
        else:
            scale_x = native_res_x / target_resolution_m
            scale_y = native_res_y / target_resolution_m

            out_w = int(round(native_w * scale_x))
            out_h = int(round(native_h * scale_y))

            out_w = max(2, out_w)
            out_h = max(2, out_h)

        dem_masked = src.read(
            1,
            out_shape=(out_h, out_w),
            resampling=resampling,
            masked=True,
        )

        dem = dem_masked.astype(float).filled(np.nan)

        transform = src.transform * src.transform.scale(
            native_w / out_w,
            native_h / out_h,
        )

    effective_res_x = native_res_x * native_w / out_w
    effective_res_y = native_res_y * native_h / out_h

    print(f"Native DEM resolution:      {native_res_x:.3f} m × {native_res_y:.3f} m")
    print(f"Requested mesh resolution:  {target_resolution_m}")
    print(f"Output DEM shape:           {out_h} × {out_w}")
    print(f"Effective mesh resolution:  {effective_res_x:.3f} m × {effective_res_y:.3f} m")
    print(f"Resampling method:          {resampling_method}")

    # ------------------------------------------------------------
    # 2. Pixel centers -> stereographic projected x/y
    # ------------------------------------------------------------
    rows, cols = np.indices(dem.shape)

    x_ps = (
        transform.c
        + (cols + 0.5) * transform.a
        + (rows + 0.5) * transform.b
    )

    y_ps = (
        transform.f
        + (cols + 0.5) * transform.d
        + (rows + 0.5) * transform.e
    )

    # ------------------------------------------------------------
    # 3. Stereographic x/y -> lunar lon/lat
    # ------------------------------------------------------------
    to_geo = Transformer.from_crs(
        dem_crs,
        geo_crs,
        always_xy=True,
    )

    lon_deg, lat_deg = to_geo.transform(x_ps, y_ps)

    lon = np.deg2rad(lon_deg)
    lat = np.deg2rad(lat_deg)

    # ------------------------------------------------------------
    # 4. lon/lat/elevation -> Moon-centered ME-like Cartesian
    # ------------------------------------------------------------
    radius = r_moon + dem

    X_me = radius * np.cos(lat) * np.cos(lon)
    Y_me = radius * np.cos(lat) * np.sin(lon)
    Z_me = radius * np.sin(lat)

    vertices_me = np.column_stack([
        X_me.ravel(),
        Y_me.ravel(),
        Z_me.ravel(),
    ])

    # ------------------------------------------------------------
    # 5. ME -> PA rotation with SPICE
    # ------------------------------------------------------------
    et = spice.str2et(utc_time)
    R_me_to_pa = spice.pxform("MOON_ME", "MOON_PA", et)

    # Row-vector convention:
    # v_pa = R_me_to_pa @ v_me for column vectors
    # therefore for row arrays:
    # vertices_pa = vertices_me @ R_me_to_pa.T
    vertices_pa = vertices_me @ R_me_to_pa.T

    valid_vertex = np.all(np.isfinite(vertices_pa), axis=1)

    # ------------------------------------------------------------
    # 6. Build triangular faces
    # ------------------------------------------------------------
    ny, nx = dem.shape
    idx = np.arange(ny * nx).reshape(ny, nx)

    faces_1 = np.column_stack([
        idx[:-1, :-1].ravel(),
        idx[1:, :-1].ravel(),
        idx[:-1, 1:].ravel(),
    ])

    faces_2 = np.column_stack([
        idx[1:, :-1].ravel(),
        idx[1:, 1:].ravel(),
        idx[:-1, 1:].ravel(),
    ])

    faces = np.vstack([faces_1, faces_2])

    # Remove triangles touching invalid DEM pixels
    keep_faces = np.all(valid_vertex[faces], axis=1)
    faces = faces[keep_faces]

    # ------------------------------------------------------------
    # 7. Build trimesh object
    # ------------------------------------------------------------
    mesh_pa = trimesh.Trimesh(
        vertices=vertices_pa,
        faces=faces,
        process=False,
    )

    print(f"Vertices:                  {len(vertices_pa):,}")
    print(f"Faces kept:                {len(faces):,}")

    aux = {
        "dem": dem,
        "x_ps": x_ps,
        "y_ps": y_ps,
        "lat_deg": lat_deg,
        "lon_deg": lon_deg,
        "X_me": X_me,
        "Y_me": Y_me,
        "Z_me": Z_me,
        "vertices_me": vertices_me,
        "vertices_pa": vertices_pa,
        "faces": faces,
        "native_resolution_x_m": native_res_x,
        "native_resolution_y_m": native_res_y,
        "effective_resolution_x_m": effective_res_x,
        "effective_resolution_y_m": effective_res_y,
        "target_resolution_m": target_resolution_m,
        "resampling_method": resampling_method,
        "r_moon": r_moon,
        "R_me_to_pa": R_me_to_pa,
        "dem_crs": dem_crs,
        "geo_crs": geo_crs,
        "transform": transform,
    }

    return mesh_pa, aux

def normalize(v):
    v = np.asarray(v, dtype=float)
    n = np.linalg.norm(v)
    if n == 0:
        raise ValueError("Cannot normalize zero vector.")
    return v / n


def build_local_frame_from_pa_mesh(mesh_pa, x_hint_pa=None):
    """
    Build a local visualization/simulation frame from a PA mesh.

    The local frame is:
        origin_local = center of the patch
        +Z_local     = radial up, away from Moon center
        +X_local     = tangent direction chosen from x_hint_pa
        +Y_local     = completes right-handed frame

    Returns:
        origin_pa
        A_local_to_pa

    where columns of A_local_to_pa are:
        [x_local_in_pa, y_local_in_pa, z_local_in_pa]
    """

    vertices = np.asarray(mesh_pa.vertices, dtype=float)
    valid = np.all(np.isfinite(vertices), axis=1)

    if not np.any(valid):
        raise ValueError("Mesh has no valid vertices.")

    # Patch center in PA coordinates
    origin_pa = np.mean(vertices[valid], axis=0)

    # Local Up = radial direction from Moon center to patch center
    z_up_pa = normalize(origin_pa)

    # Choose a global reference vector to define local X.
    # Near the south pole, PA Z is almost parallel to local up/down,
    # so [0,0,1] may be a bad choice. Use [1,0,0] by default if needed.
    if x_hint_pa is None:
        candidate = np.array([0.0, 0.0, 1.0])

        if abs(np.dot(candidate, z_up_pa)) > 0.9:
            candidate = np.array([1.0, 0.0, 0.0])
    else:
        candidate = normalize(x_hint_pa)

    # Project candidate onto the tangent plane
    x_local_pa = candidate - np.dot(candidate, z_up_pa) * z_up_pa
    x_local_pa = normalize(x_local_pa)

    # Right-handed frame: x cross y = z
    y_local_pa = np.cross(z_up_pa, x_local_pa)
    y_local_pa = normalize(y_local_pa)

    A_local_to_pa = np.column_stack([
        x_local_pa,
        y_local_pa,
        z_up_pa,
    ])

    return origin_pa, A_local_to_pa

def pa_to_local(points_pa, origin_pa, A_local_to_pa):
    """
    Convert PA coordinates to local frame.

    points_pa can be:
        shape (3,)
        shape (N, 3)
    """
    points_pa = np.asarray(points_pa, dtype=float)
    return (points_pa - origin_pa) @ A_local_to_pa


def local_to_pa(points_local, origin_pa, A_local_to_pa):
    """
    Convert local coordinates back to PA.
    """
    points_local = np.asarray(points_local, dtype=float)
    return origin_pa + points_local @ A_local_to_pa.T


def mesh_pa_to_local(mesh_pa, origin_pa, A_local_to_pa):
    """
    Convert a PA trimesh terrain mesh to local coordinates.
    Faces are unchanged.
    """
    vertices_local = pa_to_local(mesh_pa.vertices, origin_pa, A_local_to_pa)

    mesh_local = trimesh.Trimesh(
        vertices=vertices_local,
        faces=mesh_pa.faces.copy(),
        process=False,
    )

    return mesh_local

def plot_vector3d(fig, start, end, color="red", width =1, name="Vector"):
    """Aggiunge un vettore 3D (linea + punta a cono) a una figura esistente."""
    x0, y0, z0 = start
    x1, y1, z1 = end
    
    # 1. Linea del corpo del vettore
    fig.add_trace(go.Scatter3d(
        x=[x0, x1], y=[y0, y1], z=[z0, z1],
        mode='lines', line=dict(color=color, width=width), name=name
    ))
    
    # 2. Punta del vettore (Cono) posizionata esattamente sulla fine (end)
    fig.add_trace(go.Cone(
        x=[x1], y=[y1], z=[z1],                 # Posizione della punta
        u=[x1-x0], v=[y1-y0], w=[z1-z0],       # Direzione del vettore
        colorscale=[[0, color], [1, color]],    # Colore solido
        showscale=False, sizemode='absolute', sizeref=width/10
    ))

def plot_point3D(fig, point, color="red", size =1, name="Vector"):
    """Aggiunge un vettore 3D (linea + punta a cono) a una figura esistente."""
    x, y, z = point
    
    # 1. Linea del corpo del vettore
    fig.add_trace(go.Scatter3d(
        x=[x], y=[y], z=[z],
        mode='markers', marker=dict(size=12, color = color), name=name,
    ))
    

def plot_sphere3d(fig, center=(0,0,0), radius=1.0, color="lightblue", name="Sphere"):
    """Aggiunge una sfera 3D a una figura Plotly esistente."""
    xc, yc, zc = center
    
    # 1. Genera la griglia di coordinate sferiche (angoli)
    phi = np.linspace(0, np.pi, 30)
    theta = np.linspace(0, 2 * np.pi, 30)
    phi, theta = np.meshgrid(phi, theta)
    
    # 2. Trasforma in coordinate cartesiane e applica raggio e centro
    x = xc + radius * np.sin(phi) * np.cos(theta)
    y = yc + radius * np.sin(phi) * np.sin(theta)
    z = zc + radius * np.cos(phi)
    
    # 3. Aggiunge la superficie alla figura con un colore solido personalizzato
    fig.add_trace(go.Surface(
        x=x, y=y, z=z,
        colorscale=[[0, color], [1, color]],
        showscale=False,
        opacity=0.4,
        name=name
    ))

def stereographic_xy_to_local(
    x_ps_m,
    y_ps_m,
    dem_crs,
    origin_pa,
    A_local_to_pa,
    utc_time,
    alt_m=0.0,
):
    """
    Convert lunar polar stereographic coordinates to your local mesh frame.

    Inputs
    ------
    x_ps_m, y_ps_m:
        Polar stereographic DEM coordinates in meters.

    dem_crs:
        CRS of the DEM, for example src.crs.

    origin_pa, A_local_to_pa:
        Local-frame definition obtained from build_local_frame_from_pa_mesh(mesh_pa).

    utc_time:
        Epoch for SPICE MOON_ME -> MOON_PA transformation.

    alt_m:
        Height above lunar reference sphere.
        For grid-center placement, use 0.0.
        generate_grid_nodes() will later find the actual terrain height.

    Returns
    -------
    p_local:
        3D point in local coordinates.

    info:
        Dictionary with intermediate lat/lon and PA coordinates.
    """

    dem_crs = CRS.from_user_input(dem_crs)
    geo_crs = dem_crs.geodetic_crs

    if geo_crs is None:
        raise ValueError("Could not extract lunar geographic CRS from DEM CRS.")

    r_moon = float(dem_crs.ellipsoid.semi_major_metre)

    # 1. Polar stereographic x/y -> lunar lon/lat
    to_geo = Transformer.from_crs(
        dem_crs,
        geo_crs,
        always_xy=True,
    )

    lon_deg, lat_deg = to_geo.transform(x_ps_m, y_ps_m)

    lon = np.deg2rad(lon_deg)
    lat = np.deg2rad(lat_deg)

    # 2. lon/lat/alt -> ME-like Moon-centered Cartesian
    radius = r_moon + alt_m

    p_me = np.array([
        radius * np.cos(lat) * np.cos(lon),
        radius * np.cos(lat) * np.sin(lon),
        radius * np.sin(lat),
    ], dtype=float)

    # 3. ME -> PA
    et = spice.str2et(utc_time)
    R_me_to_pa = spice.pxform("MOON_ME", "MOON_PA", et)

    p_pa = R_me_to_pa @ p_me

    # 4. PA -> local
    p_local = pa_to_local(
        p_pa,
        origin_pa,
        A_local_to_pa,
    )

    info = {
        "lat_deg": lat_deg,
        "lon_deg": lon_deg,
        "p_me": p_me,
        "p_pa": p_pa,
        "p_local": p_local,
    }

    return p_local, info

In [ ]:
input_tif = r"/Users/pierazhi/Desktop/Multipath/Multipath/Refactored/tifs_new/LDEM_875S_20M.tif"

visualize_dem_2d(
    input_tif,
    downsample_factor=10,
    title="LOLA DEM 2D preview",
)

In [ ]:

bounds_m = selection_to_bounds(
    input_tif,
    method="center_latlon",
            center_lat=-89.67,
            center_lon=129.70,
            side_km=100.0,
        )
    # method="center_latlon",
    #         center_lat=-88.13,
    #         center_lon=45.91,
    #         side_km=40.0,
    #     )

crop_dem_by_bounds(
    input_tif,
    r"/Users/pierazhi/Desktop/Multipath/Multipath/Refactored/tifs_new/LDEM_875S_20M_cropped.tif",
    bounds_m,
)

visualize_dem_2d(
    r"/Users/pierazhi/Desktop/Multipath/Multipath/Refactored/tifs_new/LDEM_875S_20M_cropped.tif",
    downsample_factor=10,
    title="LOLA DEM 2D preview",
)


In [ ]:
mesh_pa, aux = dem_to_pa_mesh(
    r"/Users/pierazhi/Desktop/Multipath/Multipath/Refactored/tifs_new/LDEM_875S_20M_cropped.tif",
    target_resolution_m=400,
    utc_time="2026-06-22T00:00:00",
    resampling_method="cubic"
)

In [ ]:
pos_tx = generate_grid_nodes(mesh_pa, center=[70, -6170], n = 1, height = 50)
pos_rx = generate_grid_nodes(mesh_pa, center=[15790, -6170], n = 1, height = 10)

fig = go.Figure()
plot_sphere3d(fig, np.array([0, 0, 0]), radius = R_LOLA_M, color = "grey", name = "Luna")
v = mesh_pa.vertices
f = mesh_pa.faces

fig.add_trace(go.Mesh3d(
    x=v[:, 0],
    y=v[:, 1],
    z=v[:, 2],
    i=f[:, 0],
    j=f[:, 1],
    k=f[:, 2],
    color="dimgray",
    opacity=0.5,
    flatshading=True,
    name="Terreno Lunare",
    lighting=dict(
        ambient=0.4,
        diffuse=0.8,
        fresnel=0.2,
        specular=0.1,
        roughness=0.5
    ),
    lightposition=dict(
        x=0,
        y=2.5,
        z=-5
    )
))

# plot_point3D(fig, pos_tx, color="red", size=6, name="TX")
# plot_point3D(fig, pos_rx, color="blue", size=6, name="RX")

fig.update_layout(
    scene=dict(
        aspectmode="data",
        xaxis_title="X_PA (m)",
        yaxis_title="Y_PA (m)",
        zaxis_title="Z_PA (m)"
    )
)

fig.show()

In [ ]:
origin_pa, A_local_to_pa = build_local_frame_from_pa_mesh(mesh_pa)

mesh_local = mesh_pa_to_local(
    mesh_pa,
    origin_pa,
    A_local_to_pa,
)

with rasterio.open(r"/Users/pierazhi/Desktop/Multipath/Multipath/Refactored/tifs_new/LDEM_875S_20M_cropped.tif") as src:
    dem_crs = src.crs

utc_time = "2026-06-22T00:00:00"

pos_tx, tx_info = stereographic_xy_to_local(70,-6170, dem_crs, origin_pa, A_local_to_pa, utc_time, alt_m=0.0,)
pos_rx, tx_info = stereographic_xy_to_local( 15790,-6170, dem_crs, origin_pa, A_local_to_pa, utc_time, alt_m=0.0,)

pos_tx = np.asarray(generate_grid_nodes(mesh_local, center=pos_tx[:2], n = 1, height = 50)).squeeze()
pos_rx = np.asarray(generate_grid_nodes(mesh_local, center=pos_rx[:2], n = 1, height = 10)).squeeze()

visualize_mesh_3D(mesh_local, [pos_tx], [pos_rx], show_axes=True, show_edges=True, size = 3)

In [ ]:
boresight_tx = np.array([0, 0, -1])
boresight_rx = -boresight_tx
up_tx = np.array([0, 0, 1])
up_rx = up_tx
sep = np.linalg.norm(np.asarray(pos_tx) - np.asarray(pos_rx))


res = run_sbr_image_solver(
    mesh_local, pos_tx, pos_rx,
    boresight_tx=boresight_tx, boresight_rx=boresight_rx, up_tx=up_tx, up_rx=up_rx,
    tx_pol="RHCP", rx_pol="RHCP",
    frequenza=2.4e9, isotropic=False,
    max_bounces=3, num_rays=1e6,
    launch_mode="full_sphere", cone_aim_point=None,
    verbose=True,
    occlusion="batched",                   # ← "serial" | "batched"
    pol_convention="fixed",               # ← "fixed" | "sionna"
    max_path_distance=5*sep,
    epsilon_luna=2.87-0.01j,
    G_max_db = 12,
    hpbw_h = 75,
    hpbw_v = 75,
    enable_profiling = False,
    exact_first_order=True,
    polarization=True,
)

visualize_with_plotly(mesh_local, res, pos_tx, pos_rx, 1, True, False, False, boresight_tx, boresight_rx, up_tx, up_rx)